# FraudShield AI — Exploratory Data Analysis (notebook)

This notebook reproduces the exploratory analysis used in the project report.
It uses the **same validated loader** as the production training script, so it
will refuse to run on a corrupted or missing dataset rather than silently
producing meaningless numbers.

Model training and evaluation live in `scripts/train.py` — this notebook is for
inspection and viva demonstration only.

In [ ]:
import sys
from pathlib import Path

# Make the project importable when the notebook runs from notebooks/
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from backend.app.core.data_validation import load_dataset

df, report = load_dataset()
report

In [ ]:
df.head()

In [ ]:
counts = df["Class"].value_counts().sort_index()
print(f"Transactions : {len(df):,}")
print(f"Legitimate   : {counts[0]:,}")
print(f"Fraud        : {counts[1]:,}  ({counts[1] / len(df) * 100:.3f}%)")
print(f"Missing      : {int(df.isnull().sum().sum())}")
print(f"Time span    : {df['Time'].max() / 3600:.1f} hours")

In [ ]:
df.groupby("Class")["Amount"].describe()

## Key observations

1. The dataset is **extremely imbalanced** (~0.17% fraud), so accuracy is a
   misleading metric — the project selects models by PR-AUC instead.
2. `V1..V28` are already PCA-scaled and anonymised; only `Time` and `Amount`
   are interpretable, so only those two are standardised.
3. 1,081 exact duplicate rows exist; they are removed **before** splitting so
   no transaction can appear in both training and evaluation data.

The figures generated by `scripts/train.py` (class distribution, amount
distribution, precision-recall curves, threshold sweep, confusion matrix)
are saved under `reports/figures/`.